In [1]:
!pip install transformers peft datasets trl

In [2]:
import os
import torch
from contextlib import nullcontext
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [3]:
!pip uninstall -y bitsandbytes
!pip uninstall -y nvidia-pip nvidia-pip-cu11 nvidia-pip-cu12


In [ ]:
import os, sys
os.kill(os.getpid(), 9)


In [3]:
!pip install -U bitsandbytes


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16"  # or "float16" if GPU doesn’t support bf16
)
model_name="microsoft/Phi-3-mini-4k-instruct"
model = AutoModelForCausalLM.from_pretrained(model_name
    ,
    quantization_config=bnb_config,
    device_map="auto"  # automatically places model on GPU(s)
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [5]:
model = prepare_model_for_kbit_training(model)

In [6]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["o_proj","qkv_proj","gate_up_proj","down_proj"]
)

In [7]:
model=get_peft_model(model,lora_config)

In [10]:
from datasets import load_dataset

# Load the dataset
ds = load_dataset("Aarif1430/english-to-hindi")

# Check available splits

# Select the training split (or whichever you need)
dataset= ds["train"]

# Take only 500 examples
dataset = dataset.select(range(500))

# Verify size
print(dataset)
print(dataset[0])  # show first example

Dataset({
    features: ['english_sentence', 'hindi_sentence'],
    num_rows: 500
})
{'english_sentence': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles", 'hindi_sentence': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।'}


In [11]:
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

In [12]:
dataset = dataset.rename_column("english_sentence", "prompt")
dataset = dataset.rename_column("hindi_sentence", "completion")
dataset = dataset.map(format_dataset)
dataset = dataset.remove_columns(['prompt', 'completion', ])
messages = dataset[0]['messages']
messages

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[{'content': "However, Paes, who was partnering Australia's Paul Hanley, could only go as far as the quarterfinals where they lost to Bhupathi and Knowles",
  'role': 'user'},
 {'content': 'आस्ट्रेलिया के पाल हेनली के साथ जोड़ी बनाने वाले पेस मियामी में क्वार्टरफाइनल तक ही पहुंच सके क्योंकि इस दौर में उन्हें भूपति और नोल्स ने हराया था।',
  'role': 'assistant'}]

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

tokenizer.chat_template

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

"{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'user' %}{{'<|user|>\n' + message['content'] + '<|end|>\n'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>\n' + message['content'] + '<|end|>\n'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% else %}{{ eos_token }}{% endif %}"

In [14]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage

    gradient_checkpointing=True,

    gradient_checkpointing_kwargs={'use_reentrant': False},

    gradient_accumulation_steps=1,

    per_device_train_batch_size=16,

    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_length=256,

    packing=True,
    packing_strategy='wrapped',

    ## GROUP 3: These are typical training parameters
    num_train_epochs=2,
    learning_rate=2e-4,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='paged_adamw_8bit',

    ## GROUP 4: Logging parameters
    logging_steps=10,
    logging_dir='./logs',
    output_dir='./phi3-mini-english-hin',
    report_to='none',

    # ensures bf16 (the new default) is only used when it is actually available
    bf16=torch.cuda.is_bf16_supported(including_emulation=False)
)

In [15]:
trainer = SFTTrainer(
    model=model.base_model.model, # the underlying Phi-3 model
    peft_config=lora_config,  # added to fix issue in TRL>=0.20
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=dataset,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [16]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,1.967500


Step,Training Loss
10,1.967500
20,1.658400
30,1.606500


TrainOutput(global_step=32, training_loss=1.7367749065160751, metrics={'train_runtime': 1563.6634, 'train_samples_per_second': 0.327, 'train_steps_per_second': 0.02, 'total_flos': 2937450808737792.0, 'train_loss': 1.7367749065160751, 'entropy': 1.5992348790168762, 'num_tokens': 131004.0, 'mean_token_accuracy': 0.6079657077789307, 'epoch': 2.0})

In [18]:
def gen_prompt(tokenizer, sentence):
    converted_sample = [
        {"role": "user", "content": sentence},
    ]
    prompt = tokenizer.apply_chat_template(converted_sample,
                                           tokenize=False,
                                           add_generation_prompt=True)
    return prompt

In [22]:
sentence = "this model can translate english sentencee to hindi sentence"
prompt = gen_prompt(tokenizer, sentence)
print(prompt)

<|user|>
this model can translate english sentencee to hindi sentence<|end|>
<|assistant|>



In [31]:
sentence = "i love deeplearning "
prompt = gen_prompt(tokenizer, sentence)
print(prompt)

<|user|>
i love deeplearning <|end|>
<|assistant|>



In [32]:
def generate(model, tokenizer, prompt, max_new_tokens=264, skip_special_tokens=False):
    tokenized_input = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)

    model.eval()
    # if it was trained using mixed precision, uses autocast context
    ctx = torch.autocast(device_type=model.device.type, dtype=model.dtype) \
          if model.dtype in [torch.float16, torch.bfloat16] else nullcontext()
    with ctx:
        generation_output = model.generate(**tokenized_input,
                                           eos_token_id=tokenizer.eos_token_id,
                                           max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(generation_output,
                                    skip_special_tokens=skip_special_tokens)
    return output[0]

In [24]:
print(generate(model, tokenizer, prompt))

<|user|> this model can translate english sentencee to hindi sentence<|end|><|assistant|> इस प्रणाली का उपयोग हमें अंग्रेजी वाक्य को हिंदी वाक्य में अनुकरण करना है<|end|><|endoftext|>


In [33]:
print(generate(model, tokenizer, prompt))

<|user|> i love deeplearning <|end|><|assistant|> मुझे डीफ्लेंडिंग प्रेम है<|end|><|endoftext|>


In [ ]:
!pip install huggingface_hub
from huggingface_hub import notebook_login

notebook_login()
